In [ ]:
import pandas as pd
import pydicom
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, matthews_corrcoef
from PIL import Image
from tqdm.notebook import tqdm

In [ ]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None, reduction='mean'):
        super(BinaryFocalLoss, self).__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs, 
            targets, 
            pos_weight=self.pos_weight, 
            reduction='none'
        )
        pt = torch.exp(-bce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * bce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        else:
            return focal_loss.sum()

In [ ]:
class VinDrMLODataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        
        # Mapeamento Binário (BI-RADS 1, 2, 3 = Benigno(0) | BI-RADS 4, 5 = Maligno(1))
        self.label_map = {
            'BI-RADS 1': 0, 
            'BI-RADS 2': 0, 
            'BI-RADS 3': 0, 
            'BI-RADS 4': 1, 
            'BI-RADS 5': 1
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Caminho do DICOM
        img_path = f"{self.root_dir}/{row['study_id']}/{row['image_id']}.dicom"
        
        # Leitura do DICOM
        ds = pydicom.dcmread(img_path)
        pixel_array = ds.pixel_array.astype(float)
        
        # 1. Normalização Min-Max para 16-bits
        pixel_array = (pixel_array - np.min(pixel_array)) / (np.max(pixel_array) - np.min(pixel_array))
        pixel_array_16bit = (pixel_array * 65535).astype(np.uint16)
        
        # 2. Inicialização e Aplicação do CLAHE
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        pixel_array_clahe = clahe.apply(pixel_array_16bit)
        
        # 3. Redução para 8-bits
        pixel_array_8bit = (pixel_array_clahe / 256).astype(np.uint8)
        
        # 4. Converte para imagem PIL em RGB
        image = Image.fromarray(pixel_array_8bit).convert('RGB')
        
        lateralidade = row['laterality'] 
        
        # Espelha a mama direita para que todas fiquem orientadas como a esquerda
        if lateralidade == 'R':
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            
        # Pega a classe e converte para Binário
        label = self.label_map[row['breast_birads']]
        
        # Aplica Transformações (Tensor, Resize, Normalize...)
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Carregamento e Preparação do CSV ---
csv_path = "/backup/lucas/datasets/vindr-mammo/breast-level_annotations.csv"
images_dir = "/backup/lucas/datasets/vindr-mammo/images"

df_completo = pd.read_csv(csv_path)

# Filtra apenas MLO (Verifique se no CSV chama 'view' ou 'view_position')
df_mlo = df_completo[df_completo['view_position'] == 'MLO'].copy()

# Remove possíveis linhas sem BI-RADS anotado, se houver
df_mlo = df_mlo.dropna(subset=['breast_birads'])

# --- Split sem Data Leakage (por study_id) ---
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss1.split(df_mlo, groups=df_mlo['study_id']))

df_train = df_mlo.iloc[train_idx]
df_temp = df_mlo.iloc[temp_idx]

# 2º Passo: 10% Validação / 10% Teste
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['study_id']))

df_val = df_temp.iloc[val_idx]
df_test = df_temp.iloc[test_idx]

print(f"Total Imagens MLO: {len(df_mlo)} | Treino: {len(df_train)} | Validação: {len(df_val)} | Teste: {len(df_test)}")

# --- DataLoaders ---
BATCH_SIZE = 16 # Ajuste para 8 ou 4 se tiver erro de falta de memória de vídeo (OOM)

train_dataset = VinDrMLODataset(dataframe=df_train, root_dir=images_dir, transform=train_transform)
val_dataset = VinDrMLODataset(dataframe=df_val, root_dir=images_dir, transform=val_transform)
test_dataset = VinDrMLODataset(dataframe=df_test, root_dir=images_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

Total Imagens MLO: 9999 | Treino: 7999 | Validação: 2000


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.densenet121(weights='IMAGENET1K_V1')

num_ftrs = model.classifier.in_features

# Substituímos a camada por um bloco com Dropout e 1 neurónio (Sigmoid/Binário)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_ftrs, 1)
)
model = model.to(device)

# Peso de 15.0 para forçar o foco nos casos de cancro
pos_weight = torch.tensor([15.0]).to(device) 
criterion = BinaryFocalLoss(gamma=2.0, pos_weight=pos_weight)

# Weight Decay para evitar memorização rápida (Overfitting)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Scheduler para reduzir a taxa de aprendizagem se a validação estagnar
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=4, 
    verbose=True
)

print(device)

cuda


/home/adriano/anaconda3/envs/torch/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
num_epochs = 40
THRESHOLD = 0.50 # Limiar de aproximadamente 40% de probabilidade para ser considerado cancro

best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n--- Época {epoch+1}/{num_epochs} ---")
    
    # --- TREINAMENTO ---
    model.train()
    train_loss = 0.0
    
    loop_treino = tqdm(train_loader, desc="Treinamento", leave=False)
    
    for images, labels in loop_treino:
        images = images.to(device)
        # Converte labels para Float e matriz coluna (exigência da BCE Loss)
        labels = labels.to(device).float().view(-1, 1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        loop_treino.set_postfix(loss=loss.item())
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # --- VALIDAÇÃO ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        loop_val = tqdm(val_loader, desc="Validação", leave=False)
        for images, labels in loop_val:
            images = images.to(device)
            labels = labels.to(device).float().view(-1, 1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            # Aplica Sigmoid para obter probabilidades (0.0 a 1.0)
            probs = torch.sigmoid(outputs)
            
            # Classificação final baseada no THRESHOLD
            preds = (probs >= THRESHOLD).float()
            
            # Achata as matrizes de volta para listas simples do Scikit-Learn
            all_labels.extend(labels.view(-1).cpu().numpy())
            all_preds.extend(preds.view(-1).cpu().numpy())
            all_probs.extend(probs.view(-1).cpu().numpy())
            
    val_loss = val_loss / len(val_loader.dataset)
    
    # --- MÉTRICAS ---
    try:
        tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()
    except ValueError:
        tn = fp = fn = tp = 0
        
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.0
        
    try:
        mcc = matthews_corrcoef(all_labels, all_preds)
    except ValueError:
        mcc = 0.0
        
    print(f"Loss Treino: {train_loss:.4f} | Loss Validação: {val_loss:.4f}")
    print(f"Matriz de Confusão -> TP:{tp} | FN:{fn} | TN:{tn} | FP:{fp}")
    print(f"Sensibilidade: {sensitivity:.4f} | Especificidade: {specificity:.4f}")
    print(f"AUC-ROC: {auc:.4f} | MCC: {mcc:.4f}")
    
    # Informa o Scheduler sobre a Loss atual
    scheduler.step(val_loss)
    
    # --- SALVAMENTO ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'densenet_vindr_mlo_densenet.pth')
        print(f"🔥 Novo melhor modelo salvo! (Val Loss desceu para: {best_val_loss:.4f})")
    else:
        print(f"⚠️ Val Loss não melhorou. (Melhor até agora: {best_val_loss:.4f})")


--- Época 1/40 ---


Loss Treino: 0.6964 | Loss Validação: 0.5977
Matriz de Confusão -> TP:43 | FN:46 | TN:1334 | FP:577
Sensibilidade: 0.4831 | Especificidade: 0.6981
AUC-ROC: 0.6353
🔥 Novo melhor modelo salvo! (Val Loss desceu para: 0.5977)

--- Época 2/40 ---


Loss Treino: 0.6770 | Loss Validação: 0.5615
Matriz de Confusão -> TP:39 | FN:50 | TN:1548 | FP:363
Sensibilidade: 0.4382 | Especificidade: 0.8100
AUC-ROC: 0.6729
🔥 Novo melhor modelo salvo! (Val Loss desceu para: 0.5615)

--- Época 3/40 ---


Loss Treino: 0.6565 | Loss Validação: 0.5430
Matriz de Confusão -> TP:72 | FN:17 | TN:916 | FP:995
Sensibilidade: 0.8090 | Especificidade: 0.4793
AUC-ROC: 0.7290
🔥 Novo melhor modelo salvo! (Val Loss desceu para: 0.5430)

--- Época 4/40 ---


Loss Treino: 0.6421 | Loss Validação: 0.5112
Matriz de Confusão -> TP:63 | FN:26 | TN:1233 | FP:678
Sensibilidade: 0.7079 | Especificidade: 0.6452
AUC-ROC: 0.7448
🔥 Novo melhor modelo salvo! (Val Loss desceu para: 0.5112)

--- Época 5/40 ---


...

Loss Treino: 0.5706 | Loss Validação: 0.5365
Matriz de Confusão -> TP:58 | FN:31 | TN:1248 | FP:663
Sensibilidade: 0.6517 | Especificidade: 0.6531
AUC-ROC: 0.7191
⚠️ Val Loss não melhorou. (Melhor até agora: 0.5112)

--- Época 8/40 ---


Loss Treino: 0.5540 | Loss Validação: 0.5228
Matriz de Confusão -> TP:43 | FN:46 | TN:1737 | FP:174
Sensibilidade: 0.4831 | Especificidade: 0.9089
AUC-ROC: 0.7501
⚠️ Val Loss não melhorou. (Melhor até agora: 0.5112)

--- Época 9/40 ---


Loss Treino: 0.5134 | Loss Validação: 0.5027
Matriz de Confusão -> TP:51 | FN:38 | TN:1612 | FP:299
Sensibilidade: 0.5730 | Especificidade: 0.8435
AUC-ROC: 0.7785
🔥 Novo melhor modelo salvo! (Val Loss desceu para: 0.5027)

--- Época 10/40 ---


Loss Treino: 0.4864 | Loss Validação: 0.5416
Matriz de Confusão -> TP:50 | FN:39 | TN:1671 | FP:240
Sensibilidade: 0.5618 | Especificidade: 0.8744
AUC-ROC: 0.7683
⚠️ Val Loss não melhorou. (Melhor até agora: 0.5027)

--- Época 11/40 ---


Treinamento:   9%|▉         | 44/500 [00:44<06:16,  1.21it/s, loss=0.333]

In [ ]:
# ==========================================
# CÉLULA 7: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (E CURVA ROC)
# ==========================================

print("A carregar o melhor modelo salvo...")
model.load_state_dict(torch.load('densenet_vindr_mlo_densenet.pth', weights_only=True))
model.eval()

all_labels_test = []
all_probs_test = []

print("A extrair probabilidades do conjunto de TESTE...")
with torch.no_grad():
    loop_test = tqdm(test_loader, desc="Avaliando (Teste)", leave=False)
    for images, labels in loop_test:
        images = images.to(device)
        labels = labels.to(device).float().view(-1, 1)
        
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        
        all_labels_test.extend(labels.view(-1).cpu().numpy())
        all_probs_test.extend(probs.view(-1).cpu().numpy())

# --- CÁLCULO DA CURVA ROC E AUC ---\n
fpr, tpr, thresholds = roc_curve(all_labels_test, all_probs_test)
auc_final = roc_auc_score(all_labels_test, all_probs_test)

# Cálculo com o THRESHOLD base de 0.50 para o MCC final
preds_test = (np.array(all_probs_test) >= 0.50).astype(int)
mcc_final = matthews_corrcoef(all_labels_test, preds_test) if len(np.unique(all_labels_test)) > 1 else 0.0

# --- ENCONTRAR O LIMIAR PARA 90% DE SENSIBILIDADE ---
idx_alvo = np.where(tpr >= 0.90)[0][0] # Ajustado para 90%
limiar_ideal = thresholds[idx_alvo]
especificidade_alvo = 1 - fpr[idx_alvo]

# --- PLOT DA CURVA ROC ---
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {auc_final:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

plt.scatter(
    fpr[idx_alvo], tpr[idx_alvo], 
    color='red', s=100, zorder=5, 
    label=f'Alvo: Sensibilidade 90%'
)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos (1 - Especificidade)', fontsize=12)
plt.ylabel('Taxa de Verdadeiros Positivos (Sensibilidade)', fontsize=12)
plt.title('Curva ROC - DenseNet121 (Vista MLO) - Teste', fontsize=14)
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)
plt.show()

# --- RELATÓRIO CLÍNICO ---
print(f"--- RESULTADOS FINAIS (CONJUNTO DE TESTE) ---")
print(f"AUC-ROC Global: {auc_final:.4f}")
print(f"MCC Global (Threshold 0.5): {mcc_final:.4f}")
print(f"\nPara garantir o cenário clínico de ~90% de Sensibilidade:")
print(f"  -> O Limiar (Threshold) ideal é: {limiar_ideal:.4f}")
print(f"  -> Com este limiar, a sua Especificidade será: {especificidade_alvo:.4f}")